**Question 1**

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MaskedConvLayer(nn.Module):
    def __init__(self, kernel_size):
        super(MaskedConvLayer, self).__init__()
        self.kernel_size = kernel_size
        # تعریف پارامترهای یادگیری
        self.W = nn.Parameter(torch.randn(1, 1, kernel_size, kernel_size))
        self.M = nn.Parameter(torch.randn(1, 1, kernel_size, kernel_size))

    def forward(self, x):
        # حاصل‌ضرب هادامار فیلتر و ماسک
        effective_weight = self.W * self.M

        # پیاده‌سازی دستی کانولوشن با unfold
        x_unfold = F.unfold(x, kernel_size=self.kernel_size)
        weight_flat = effective_weight.view(1, -1)

        # ضرب ماتریسی برای محاسبه خروجی
        out_unfold = weight_flat @ x_unfold

        out_size = x.size(-1) - self.kernel_size + 1
        return out_unfold.view(x.size(0), 1, out_size, out_size)

# ۱. ایجاد داده‌های تصادفی طبق صورت سوال
torch.manual_seed(42) # برای نتایج مشابه
input_tensor = torch.randn(1, 1, 5, 5)
target_tensor = torch.randn(1, 1, 3, 3)

# ۲. تعریف مدل و تابع خطا
model = MaskedConvLayer(kernel_size=3)
criterion = nn.MSELoss()

# ۳. گام رفت (Forward Pass)
output = model(input_tensor)
loss = criterion(output, target_tensor)

# ۴. گام برگشت (Backward Pass)
loss.backward()

# ۵. نمایش نتایج
print(f"Initial Loss: {loss.item():.4f}")
print("-" * 30)
print("Gradients for W (first 3 elements):")
print(model.W.grad.view(-1)[:3])
print("\nGradients for M (first 3 elements):")
print(model.M.grad.view(-1)[:3])

# بررسی صحت: گرادیان W باید متناسب با مقادیر M و گرادیان M متناسب با W باشد.

Initial Loss: 4.6200
------------------------------
Gradients for W (first 3 elements):
tensor([ 1.1261, -1.0485,  0.1057])

Gradients for M (first 3 elements):
tensor([ 0.4340, -0.3002, -1.0274])


**Question 2**

In [10]:
import torch
import torch.nn as nn

class Bottleneck(nn.Module):
    def __init__(self, in_channels, mid_channels, out_channels):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False)
        self.conv2 = nn.Conv2d(mid_channels, mid_channels, kernel_size=3, padding=1, bias=False)
        self.conv3 = nn.Conv2d(mid_channels, out_channels, kernel_size=1, bias=False)

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        return out

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ایجاد مدل‌ها
input_tensor = torch.randn(1, 256, 14, 14)
model_64 = Bottleneck(256, 64, 256)
model_32 = Bottleneck(256, 32, 256)

# ۱. چاپ تعداد پارامترها و مقایسه با محاسبات تحلیلی
print(f"Parameters (Mid=64): {count_parameters(model_64)} | Calculated: 69632")
print(f"Parameters (Mid=32): {count_parameters(model_32)} | Calculated: 25600")

# ۲. بررسی یکسان بودن Shape خروجی‌ها
out64 = model_64(input_tensor)
out32 = model_32(input_tensor)

print(f"Output Shape (64): {out64.shape}")
print(f"Output Shape (32): {out32.shape}")
print(f"Match: {out64.shape == out32.shape}")

Parameters (Mid=64): 69632 | Calculated: 69632
Parameters (Mid=32): 25600 | Calculated: 25600
Output Shape (64): torch.Size([1, 256, 14, 14])
Output Shape (32): torch.Size([1, 256, 14, 14])
Match: True
